# test — how often one course is followed by another

Set `COURSE` and `COURSE_CONTINUATION` below and run the notebook: for every cohort it counts
how often a student took `COURSE` and then `COURSE_CONTINUATION` in the **following** semester.

Counted per event pair, not per student — a student who does it twice contributes 2. The log is
`students_filtered.csv` (complete data and finished studies), split into cohorts on
`case:in_time` the way `00_preprocess.ipynb` does.

In [ ]:
from pathlib import Path

import pandas as pd

LOG_PATH = Path.cwd() / "data" / "preprocessed" / "students_filtered.csv"

# --- what to count: any two activity tokens, outcome included
COURSE = "Op Sys (fail)"
COURSE_CONTINUATION = "Op Sys (fail )"

COHORTS = {True: "on-time", False: "late"}

log = pd.read_csv(LOG_PATH)

# `time:timestamp` is a semester, "YYYY/1" or "YYYY/2" -- as a running integer
# "the next semester" is simply +1
year, part = log["time:timestamp"].str.split("/", expand=True).astype(int).T.values
log["semester"] = year * 2 + (part - 1)
log["cohort"] = log["case:in_time"].map(COHORTS)

# a typo would otherwise just count zero, which reads like a finding
for token in (COURSE, COURSE_CONTINUATION):
    assert token in set(log["concept:name"]), f"{token!r} is not an activity in the log"

In [2]:
occurrences = log.loc[
    log["concept:name"] == COURSE, ["case:concept:name", "cohort", "semester"]
]
continuations = set(
    map(
        tuple,
        log.loc[
            log["concept:name"] == COURSE_CONTINUATION,
            ["case:concept:name", "semester"],
        ].values,
    )
)

# an occurrence counts when the SAME student has the continuation one semester later
followed = occurrences[
    [(student, s + 1) in continuations for student, _, s in occurrences.values]
]

counts = (
    pd.DataFrame(
        {
            COURSE: occurrences.groupby("cohort").size(),
            "followed by " + COURSE_CONTINUATION: followed.groupby("cohort").size(),
        }
    )
    .reindex(list(COHORTS.values()))
    .fillna(0)
    .astype(int)
)
counts["share"] = counts.iloc[:, 1].div(counts.iloc[:, 0]).fillna(0)

print(f"{COURSE!r} followed by {COURSE_CONTINUATION!r} in the next semester")
display(counts.style.format({"share": "{:.1%}"}))

'Op Sys (fail)' followed by 'Op Sys (fail)' in the next semester


,Op Sys (fail),followed by Op Sys (fail),share
cohort,,,
on-time,5,1,20.0%
late,41,2,4.9%
